# TensorFlow & Keras Fundamentals
**HackerRank Prep — Theme 0.7**

## TL;DR
TensorFlow 2.x with the Keras API is the framework behind some of the most impactful computational biology models:
- **Enformer** (DeepMind) — predicts gene expression from 200kb DNA windows
- **Basenji / Baskerville** (Google) — regulatory genomics at single-nucleotide resolution
- **DeepVariant** (Google) — clinical-grade variant calling from sequencing reads
- **AlphaFold2** original — protein structure prediction (JAX, but same paradigm)

Even if your daily driver is PyTorch, you WILL encounter TensorFlow codebases in biology. This notebook makes you fluent in both.

---

## Learning Objectives
- [ ] Create `tf.Tensor` and `tf.Variable`; use `tf.GradientTape` for differentiation
- [ ] Build models with Keras Sequential and Functional API
- [ ] Write custom layers by subclassing `tf.keras.layers.Layer`
- [ ] Train models using `model.compile()` + `model.fit()` with callbacks
- [ ] Write custom training loops with `tf.GradientTape`
- [ ] Build efficient data pipelines with `tf.data.Dataset`
- [ ] Use mixed precision and `tf.function` for performance
- [ ] Implement a 1D CNN for DNA sequences (Enformer-style)
- [ ] Translate fluently between PyTorch and TensorFlow patterns

---

## Self-Check After Completing This Notebook
1. What is `tf.function` and when does tracing happen? What Python operations cause retracing?
2. Compare `tf.GradientTape` to PyTorch's autograd. What is recorded and when is it cleared?
3. Write the Keras Functional API pattern for a model with two inputs and one output.
4. Why must you call `tf.data.Dataset.prefetch(tf.data.AUTOTUNE)` in production pipelines?
5. When would you choose TensorFlow over PyTorch for a biology project?

## Beginner Teaching Frame

**Who should fully work through this notebook:** students with little or no ML background.

**How to study it on a first pass:**
- read every markdown section before touching the code
- run the code in small chunks
- paraphrase each concept in your own words
- write one tiny example from memory after finishing

**Common traps:** memorizing syntax without understanding the data flow, skipping probability, and rushing past train/test split discipline.


## Canonical Resource Upgrade

If you need a second explanation, use these first:
- [CS50P](https://cs50.harvard.edu/python/) for programming fundamentals
- [Harvard Stat 110](https://stat110.hsites.harvard.edu/) for probability intuition
- [scikit-learn MOOC](https://inria.github.io/scikit-learn-mooc/) for correct ML workflow
- [PyTorch Tutorials](https://docs.pytorch.org/tutorials/) once you reach the deep learning notebooks


## Prerequisites & Learning Path

| | |
|--|--|
| Prerequisites | 00/06 PyTorch Fundamentals — compare frameworks; OR standalone for genomics ML track |
| You Are Here | Module 00/07 — TensorFlow/Keras |
| Next Steps | 07/01 AlphaFold3 Architecture (uses JAX, related ecosystem) |
| Goal | Build and train Keras models for genomics; understand tf.function, GradientTape, TF Hub |

### From Scratch? Start Here:
1. [MIT 6.S191 Deep Learning (free)](http://introtodeeplearning.com) — annual course, TensorFlow-based
2. [TensorFlow tutorials (free)](https://www.tensorflow.org/tutorials) — official, comprehensive
3. This notebook

**Time:** 10-15 hours | **Difficulty:** ⭐⭐⭐⭐

### Cross-References
- Builds on: 00/06 PyTorch Fundamentals (conceptual comparison), 00/02 ML Fundamentals
- Parallel to: 00/06 PyTorch (compare frameworks)
- Used by: Genomics ML track (Enformer-style models in 02/01, 04/01)

## 🤖 AI Walkthrough — Claude Prompts

Open a Claude conversation alongside this notebook and use these prompts:

**On tf.GradientTape:**
```
Explain tf.GradientTape to me. How is it different from PyTorch's autograd?
What does 'recording' mean inside the tape context? When is the tape cleared?
When should I use persistent=True and why does it have a memory cost?
```

**On tf.function:**
```
Explain tf.function and 'tracing' in TensorFlow.
What is a 'concrete function' vs a 'Python function'?
Why does printing inside tf.function only execute once (at trace time) not every call?
What Python operations cause the function to retrace? How do I avoid retracing overhead?
```

**On the Keras Functional API:**
```
Explain the Keras Functional API.
Why would I use Functional API instead of Sequential?
Show me a multi-input, multi-output model for protein sequence + structure features.
What is the difference between a Keras Model and a Keras Layer?
```

**On tf.data pipelines:**
```
Explain the tf.data pipeline: Dataset.from_tensor_slices vs from_generator vs TFRecord.
What does .map() do vs .batch() vs .prefetch()? In what order should they be called?
Why does prefetch(AUTOTUNE) speed up training?
```

**On Enformer / genomics CNNs:**
```
Explain the Enformer architecture for predicting gene expression from DNA.
What is the input? What is the output? Why does it use 1D convolutions followed by
attention? How does the receptive field of a dilated CNN compare to a regular CNN?
```

**On PyTorch vs TensorFlow choice:**
```
When should I use PyTorch vs TensorFlow for a bioinformatics project?
Consider: existing pretrained models, deployment (TFServing vs TorchServe),
research flexibility, ecosystem (HuggingFace, PyG, TFHub).
```

---

## Interview Tip Bank
> **tf.function gotcha**: Python side effects (print, append to list) execute only once at trace time, not on every call. Bugs that only trigger at runtime are invisible to tracing.

> **GradientTape vs autograd**: In TensorFlow, you control the tape scope explicitly. In PyTorch, the graph is built implicitly as operations execute. PyTorch's approach is easier to debug; TF's `tf.function` enables XLA compilation.

> **model.fit() vs custom loop**: `model.fit()` is great for standard training but hides the loop. Use a custom loop when you need: multi-step gradient updates, custom metric aggregation, differential loss weights per batch, or debugging gradient values.

> **TFRecord format**: For large genomics datasets (Enformer uses 196kb DNA windows × millions of regions), TFRecord + tf.data is ~10x faster than reading individual files per sample.

## Prerequisites + Comparison to PyTorch

Before this notebook:
- **Notebook 00/01**: Python, NumPy — both frameworks use array operations
- **Notebook 06** (PyTorch Fundamentals) — this notebook assumes familiarity with Notebook 06; many comparisons are explicit

### TensorFlow vs PyTorch — High-Level Map
| Concept | PyTorch | TensorFlow/Keras |
|---------|---------|------------------|
| Tensor with grad | `x = torch.tensor(..., requires_grad=True)` | `x = tf.Variable(...)` |
| Gradient computation | Implicit (autograd builds graph as you go) | Explicit: `with tf.GradientTape() as tape:` |
| Model building | `class MyNet(nn.Module)` | `tf.keras.layers.Layer` or Keras API |
| Sequential model | `nn.Sequential(...)` | `tf.keras.Sequential([...])` |
| Training | Manual loop | `model.fit()` or manual loop with tape |
| Loss functions | `nn.CrossEntropyLoss()` | `tf.keras.losses.SparseCategoricalCrossentropy()` |
| Optimizers | `torch.optim.Adam(model.parameters())` | `tf.keras.optimizers.Adam()` |
| Data loading | `DataLoader(dataset)` | `tf.data.Dataset` |
| Save/load | `torch.save(state_dict)` | `model.save('path')` (SavedModel) |
| GPU placement | `model.to(device)` | Automatic (or `with tf.device('/GPU:0'):`) |
| Static graph / JIT | `torch.jit.script()` | `@tf.function` |
| Mixed precision | `torch.autocast` + `GradScaler` | `tf.keras.mixed_precision.set_global_policy('mixed_float16')` |

### Install check
```bash
pip install tensorflow       # TF 2.x (CPU + GPU)
pip install tensorflow-metal # macOS Apple Silicon GPU support
```

## Section 1 — TensorFlow Tensors & Eager Execution

### Eager Execution (TF 2.x default)
In TF 2.x, operations execute **immediately** — just like NumPy or PyTorch. This is called eager execution. TF 1.x required building a static graph first and running it in a Session; that paradigm is gone in TF 2.x.

### `tf.constant` vs `tf.Variable`
- `tf.constant`: Immutable. Cannot be updated. Used for fixed data (inputs, hyperparams).
- `tf.Variable`: Mutable. Tracked by `tf.GradientTape`. Used for model parameters and any state that changes during training.

### `tf.GradientTape`
The TF equivalent of PyTorch's autograd. You explicitly open a `with tf.GradientTape() as tape:` context. Inside this context, TF records all operations on `tf.Variable` objects (and on `tf.constant` if you call `tape.watch(tensor)`).

After the context exits, call `tape.gradient(loss, variables)` to get gradients. **The tape is consumed after one call** — use `persistent=True` if you need multiple gradient calls.

In [ ]:
# TensorFlow Tensors & Eager Execution
import numpy as np
try:
    import tensorflow as tf
    print(f"TensorFlow version: {tf.__version__}")

    # TF tensors
    x = tf.constant([[1.0, 2.0], [3.0, 4.0]])
    print(f"Tensor: {x}")
    print(f"Shape: {x.shape}, dtype: {x.dtype}")

    # Eager execution: operations run immediately (like numpy)
    y = tf.matmul(x, x)
    print(f"x @ x = {y.numpy()}")

    # GradientTape: manual gradient computation
    w = tf.Variable([1.0, 2.0, 3.0])
    with tf.GradientTape() as tape:
        y = tf.reduce_sum(w**2)  # y = sum(w^2)
    grad = tape.gradient(y, w)
    print(f"dy/dw = 2*w = {grad.numpy()}")

except ImportError:
    print("TensorFlow not installed. Key concepts:")
    print("  tf.constant: immutable tensor")
    print("  tf.Variable: mutable tensor (trainable weights)")
    print("  tf.GradientTape: records ops for auto-differentiation")
    print("  Eager execution: ops run immediately (no graph construction)")

    # Demonstrate concepts with numpy
    import numpy as np
    w = np.array([1.0, 2.0, 3.0])
    y = np.sum(w**2)
    grad = 2 * w  # dy/dw analytically
    print(f"\nNumpy analog: w={w}, y={y:.1f}, grad={grad}")

## Section 2 — Keras Sequential & Functional API

### Three Ways to Build a Keras Model
1. **Sequential API**: Simple linear stack of layers. Least flexible but most readable.
2. **Functional API**: Define layers as function calls on tensors. Supports branching, multiple inputs/outputs, shared layers.
3. **Model subclassing**: Full Python class, like PyTorch's `nn.Module`. Most flexible.

### When to use each:
| API | Use when |
|-----|----------|
| Sequential | Linear pipeline (input → hidden → output) with no branches |
| Functional | Multi-input, multi-output, skip connections (ResNets), shared weights |
| Subclassing | Dynamic computation, non-standard loops, research |

### Custom Layers
Subclass `tf.keras.layers.Layer` to create reusable components with learnable weights. Two key methods:
- `build(input_shape)`: Create weights (called once, lazily, on first call)
- `call(inputs)`: The forward computation (called every time the layer is used)

In [ ]:
# Keras Sequential vs Functional API
import numpy as np
try:
    import tensorflow as tf
    from tensorflow import keras

    # Sequential API: linear stack of layers
    seq_model = keras.Sequential([
        keras.layers.Dense(64, activation='relu', input_shape=(20,)),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(3, activation='softmax')
    ], name='sequential_model')

    # Functional API: allows branches, skip connections
    inputs = keras.Input(shape=(20,))
    x = keras.layers.Dense(64, activation='relu')(inputs)
    x = keras.layers.Dropout(0.3)(x)
    # Residual connection
    skip = keras.layers.Dense(32)(inputs)
    x = keras.layers.Dense(32, activation='relu')(x)
    x = keras.layers.Add()([x, skip])
    outputs = keras.layers.Dense(3, activation='softmax')(x)
    func_model = keras.Model(inputs, outputs, name='functional_model')

    print("Sequential model:")
    seq_model.summary(line_length=60)
    print("\nFunctional model (with skip connection):")
    func_model.summary(line_length=60)

except ImportError:
    print("TensorFlow not installed. Sequential vs Functional API concepts:")
    print("  Sequential: simple linear stack — model.add(layer)")
    print("  Functional: DAG of layers — enables skip connections, multi-input")
    print("  Subclassing: full Python control — class MyModel(tf.keras.Model)")
    print()
    print("Use Sequential for simple stacks")
    print("Use Functional for ResNets, U-Nets, multi-branch architectures")
    print("Use Subclassing for custom training logic (like AlphaFold)")

## Section 3 — Training with `model.fit()` vs Custom Training Loops

### `model.compile()` + `model.fit()` — The High-Level Path
For standard supervised learning, Keras's built-in training is convenient and optimized:
1. `model.compile(optimizer, loss, metrics)` — configure training
2. `model.fit(X_train, y_train, ...)` — run the training loop

### Callbacks — Automate Training Management
Callbacks hook into the training loop at specific points (epoch start/end, batch start/end):

| Callback | Purpose |
|----------|---------|
| `EarlyStopping` | Stop when val metric stops improving |
| `ModelCheckpoint` | Save best model weights during training |
| `ReduceLROnPlateau` | Reduce LR when loss plateaus |
| `TensorBoard` | Log metrics, histograms, graphs for visualization |
| Custom callback | Any custom logic (e.g., log to W&B, send Slack alerts) |

### Custom Training Loop — The Low-Level Path
When `model.fit()` is too limiting, you take full control with `tf.GradientTape`. Use when:
- Multiple loss terms with different update frequencies
- Custom gradient clipping logic per parameter group
- Debugging NaN gradients interactively
- Implementing techniques like gradient reversal (domain adaptation)

In [ ]:
# Training with model.fit() and Callbacks
import numpy as np
try:
    import tensorflow as tf
    from tensorflow import keras

    np.random.seed(42)
    X = np.random.randn(500, 20).astype(np.float32)
    y = (X[:, 0] + X[:, 1] > 0).astype(np.int32)

    model = keras.Sequential([
        keras.layers.Dense(32, activation='relu', input_shape=(20,)),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
    ]

    history = model.fit(X, y, validation_split=0.2, epochs=50, batch_size=32,
                        callbacks=callbacks, verbose=0)

    stopped_epoch = len(history.history['loss'])
    best_val_loss = min(history.history['val_loss'])
    print(f"Training stopped at epoch {stopped_epoch} (early stopping)")
    print(f"Best val_loss: {best_val_loss:.4f}")
    print(f"Final val_accuracy: {history.history['val_accuracy'][-1]:.3f}")

except ImportError:
    print("TensorFlow not installed. model.fit() key arguments:")
    print("  validation_split=0.2  — last 20% of data for validation")
    print("  callbacks=[EarlyStopping, ReduceLROnPlateau, ModelCheckpoint]")
    print("  EarlyStopping(patience=5) — stop when val_loss doesn't improve")
    print("  restore_best_weights=True — revert to best checkpoint")

## Section 4 — Key TensorFlow Patterns

### `tf.data.Dataset` — The Production Data Pipeline
For biology datasets (Enformer uses 40GB of TFRecord files), raw Python loops as data loaders are too slow. `tf.data` solves this:
- **`from_tensor_slices`**: In-memory data already loaded as NumPy/TF tensors
- **`from_generator`**: Any Python iterable (FASTA files, HDF5, etc.)
- **TFRecord + `TFRecordDataset`**: Binary serialized format, best throughput for large datasets

**Key pipeline pattern:**
```python
dataset = (
    tf.data.Dataset.from_tensor_slices(data)
    .map(preprocess_fn, num_parallel_calls=tf.data.AUTOTUNE)  # Parallel preprocessing
    .shuffle(buffer_size=1000)                                  # Shuffle
    .batch(32)                                                  # Batch
    .prefetch(tf.data.AUTOTUNE)                                 # Load next batch while GPU trains
)
```

### `tf.function` — Graph Compilation
`@tf.function` converts a Python function into a TF graph (like PyTorch's `torch.jit.script` but automatic). Benefits: XLA compilation, graph optimization, faster execution on GPU.

Caveat: **Python side effects are only executed at trace time**, not on every call. Never use Python `print()` for debugging inside `@tf.function` — use `tf.print()` instead.

In [ ]:
# tf.data Pipeline for Efficient Training
import numpy as np
try:
    import tensorflow as tf

    # Create dataset from numpy arrays
    X = np.random.randn(1000, 20).astype(np.float32)
    y = (X[:, 0] > 0).astype(np.float32)

    # tf.data pipeline: efficient data loading
    BATCH_SIZE = 32
    AUTOTUNE = tf.data.AUTOTUNE

    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    train_ds = (dataset
                .shuffle(buffer_size=200, seed=42)
                .batch(BATCH_SIZE)
                .prefetch(AUTOTUNE))  # overlap CPU preprocessing with GPU training

    print(f"Dataset element spec: {train_ds.element_spec}")

    # One batch
    for xb, yb in train_ds.take(1):
        print(f"Batch X: {xb.shape}, y: {yb.shape}")

    # Count batches
    n_batches = sum(1 for _ in train_ds)
    print(f"Total batches: {n_batches} (1000 samples / {BATCH_SIZE})")

    # Key: prefetch overlaps data preparation with model training
    print("\ntf.data benefits: prefetch, parallel map, memory-efficient")

except ImportError:
    print("TensorFlow not installed. tf.data key operations:")
    print("  .from_tensor_slices(X, y) — numpy to dataset")
    print("  .shuffle(buffer_size) — randomize order")
    print("  .batch(32) — create mini-batches")
    print("  .prefetch(AUTOTUNE) — pipeline CPU/GPU overlap")
    print("  .map(fn, num_parallel_calls) — parallel preprocessing")

## Section 5 — TensorFlow for Genomics & Biology

### DNA Sequence as Input
DNA sequences (A, T, G, C) are one-hot encoded before feeding to neural networks:
```
A → [1, 0, 0, 0]
T → [0, 1, 0, 0]
G → [0, 0, 1, 0]
C → [0, 0, 0, 1]
```
For a 1000 bp sequence: input shape = `(1000, 4)`, which is treated as a 1D time series for `Conv1D`.

### Enformer Architecture Overview
Enformer (Avsec et al., 2021) takes 196,608 bp of DNA and predicts 5,313 genomic tracks (ATAC-seq, ChIP-seq, RNA-seq) at 128 bp resolution.

**Architecture**:
1. Conv tower: Conv1D(768, kernel=15) + 6 residual Conv1D blocks (GELU activation, BatchNorm)
2. Dilated residual conv tower: exponentially increasing dilation rates (1, 2, 4, 8, ...)
3. Transformer tower: 11 transformer blocks (MultiHeadAttention + FFN)
4. Crop center 896 positions
5. PointwiseLinear head: Dense(512) → Dense(n_targets)

### Basset / Basenji Architecture
For shorter windows (1-2kb), a pure CNN with:
- Conv1D blocks + MaxPooling1D + BatchNorm
- Skip connections (ResNet-style)
- Final Dense layers for multi-task output

In [ ]:
# TensorFlow/Keras summary: key patterns
import numpy as np
try:
    import tensorflow as tf
    print(f"TF version: {tf.__version__}")
    # Simple regression model
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(16, activation='relu', input_shape=(5,)),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    X = np.random.randn(100, 5).astype(np.float32)
    y = X.sum(1, keepdims=True)
    model.fit(X, y, epochs=10, verbose=0)
    print(f"Final loss: {model.evaluate(X, y, verbose=0):.4f}")
except ImportError:
    print("Key Keras patterns:")
    print("  model.fit(X, y, epochs=N, validation_split=0.2, callbacks=[...])")
    print("  model.evaluate(X_test, y_test) → loss, metrics")
    print("  model.predict(X_new) → predictions")
    print("  model.save('model.h5') / keras.models.load_model('model.h5')")

## Section 6 — PyTorch vs TensorFlow Comparison

### Decision Framework — When to Use Each

| Criterion | PyTorch | TensorFlow/Keras |
|-----------|---------|------------------|
| Research & prototyping | **Better** — dynamic graph, easier debugging | Good |
| Existing pretrained models (proteins, molecules) | **Better** — ESM2, OpenFold, PyG all PyTorch | Enformer, Basenji, DeepVariant |
| HuggingFace Transformers | **Native** — PyTorch first | TF supported but secondary |
| Production serving | Good (TorchServe, ONNX) | **Better** — TF Serving, TFLite |
| Mobile / embedded | Limited | **Better** — TFLite |
| TPU training at scale | Limited | **Better** — native TPU support |
| Graph Neural Networks | **Better** — PyTorch Geometric (PyG) | Limited (not well supported) |
| Genomics / regulatory biology | Both (PyTorch gaining) | **Enformer, Basenji, DeepVariant** |
| AlphaFold-family models | **Better** — OpenFold, ESMFold in PyTorch | AlphaFold2 original in JAX |

### For This Curriculum (bioinformatics ML)
- **Default choice**: PyTorch (ESM2, OpenFold, Module 10 capstone)
- **Learn TensorFlow**: For Enformer/Basenji codebases; many genomics papers
- **Learn both**: Maximizes your ability to contribute to any biology codebase

In [ ]:
# TensorFlow/Keras summary: key patterns
import numpy as np
try:
    import tensorflow as tf
    print(f"TF version: {tf.__version__}")
    # Simple regression model
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(16, activation='relu', input_shape=(5,)),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    X = np.random.randn(100, 5).astype(np.float32)
    y = X.sum(1, keepdims=True)
    model.fit(X, y, epochs=10, verbose=0)
    print(f"Final loss: {model.evaluate(X, y, verbose=0):.4f}")
except ImportError:
    print("Key Keras patterns:")
    print("  model.fit(X, y, epochs=N, validation_split=0.2, callbacks=[...])")
    print("  model.evaluate(X_test, y_test) → loss, metrics")
    print("  model.predict(X_new) → predictions")
    print("  model.save('model.h5') / keras.models.load_model('model.h5')")

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- Static vs dynamic computation graphs: TensorFlow 1.x graph mode vs TF2 eager, XLA compilation model
- `tf.function` tracing semantics: AutoGraph, Python side effects in traced functions, retracing triggers
- Keras layer internals: `build()` vs `call()`, `trainable_weights` vs `non_trainable_weights`
- Key paper: Abadi et al. (2016) "TensorFlow: A System for Large-Scale Machine Learning" (OSDI)

### 2️⃣ Must-Have Popular Resources
#### 🟢 Start Here (Zero TensorFlow Background)
- 🆓 **TensorFlow Official Tutorials** — [tensorflow.org/tutorials](https://www.tensorflow.org/tutorials) — free, step-by-step
- 🆓 **Kaggle Intro to Deep Learning (Keras)** — [kaggle.com/learn/intro-to-deep-learning](https://www.kaggle.com/learn/intro-to-deep-learning) — free, 4h
- 🆓 **MIT 6.S191 TF Labs** — [introtodeeplearning.com](http://introtodeeplearning.com/) — hands-on TF labs, free
- 🆓 **Hands-On ML (Géron) — preview chapters** — [github.com/ageron/handson-ml3](https://github.com/ageron/handson-ml3) — free Jupyter notebooks
- 📘 Book: *Hands-On Machine Learning* ch. 10-19 (Géron, 3rd ed.) — TF2/Keras chapters
- 🎓 Course: MIT 6.S191 Deep Learning — [introtodeeplearning.com](http://introtodeeplearning.com) free, annual updates, TensorFlow-based
- 🎓 Course: TensorFlow official tutorials — [tensorflow.org/tutorials](https://www.tensorflow.org/tutorials), comprehensive
- ⭐ GitHub: [keras-team/keras](https://github.com/keras-team/keras) — 62k★ (Keras 3 multi-backend)
- 📊 Kaggle: Intro to Deep Learning (Keras-based) + genomics signal prediction competitions

### 3️⃣ Practicals — Hands-On
- 💻 Exercise: TF Hub transfer learning — image classification with EfficientNet in < 20 lines
- 💻 Exercise: Keras functional API multi-input model — combine sequence + structure features
- 🔗 GitHub: Build a DeepVariant-style 1D CNN for DNA variant calling
- 🤗 HuggingFace: [deepmind/enformer](https://huggingface.co/deepmind/enformer) — Enformer gene expression model
- 📊 Kaggle: Genomic signal prediction (ENCODE ChIP-seq peaks classification)

### 4️⃣ Real-World Problems
- 🏭 Industry: Enformer (DeepMind), Basenji/Baskerville (Google Brain), DeepVariant (Google) — all TensorFlow
- 📊 Dataset: ENCODE functional genomics — on [encodeproject.org](https://www.encodeproject.org), TF binding site prediction
- 🔬 Application: Regulatory sequence modelling — CNN on DNA one-hot encoding to predict TF binding

### 5️⃣ Interview Tips
- ❓ Must know: `tf.function` retracing trap — Python control flow and side effects cause retraces, use `tf.cond`/`tf.while_loop`
- ❓ Must know: channels-first (`NCHW`) vs channels-last (`NHWC`) — GPU performance difference, TPU requires channels-last
- ⚠️ Common mistake: `GradientTape` with `persistent=False` (default) — tape is consumed after first `gradient()` call
- ⚠️ Common mistake: mixing Keras `.fit()` with custom training loops — choose one approach per model
- 💡 Pro tip: Keras 3.0 runs on TF, JAX, or PyTorch backend — frame this as "write once, deploy anywhere"

### 6️⃣ Hot Industry Topics
- 🔥 Trending: JAX replacing TensorFlow at Google DeepMind — [google/jax](https://github.com/google/jax) 30k★
- 🔥 Trending: Keras 3.0 multi-backend — [keras-team/keras](https://github.com/keras-team/keras) truly backend-agnostic
- 🔥 Trending: [google/flax](https://github.com/google/flax) 6k★ — JAX neural network library used for protein models
- 🚀 Build this: Replicate Enformer's input processing in Keras — DNA sequence → one-hot → 1D CNN → gene expression predictions

## Interview Q&A — TensorFlow/Keras

**Q: What is the difference between Sequential, Functional, and Subclassing APIs?**
A: Sequential: stack of layers, simplest, no branching. Functional: explicit input/output tensors, supports multi-input/output and DAG architectures. Subclassing: full Python class, most flexible, used for custom training loops. For protein models: functional API for dual-input (sequence + structure), subclassing for custom training.

**Q: When should you use `model.fit()` vs a custom training loop?**
A: `model.fit()` for standard supervised learning — handles batching, callbacks, metrics automatically. Custom loop (`tf.GradientTape`) for: multi-task loss with different LR per component, RL training, diffusion model denoising, anything requiring custom gradient manipulation.

**Q: What is tf.data and why does it matter for genomics?**
A: `tf.data.Dataset` is a lazy pipeline for loading data. For genomics: genome files are GBs, can't fit in RAM. `tf.data` streams from disk with prefetching and parallel mapping. Use `.cache()` if data fits in memory after first epoch, `.prefetch(tf.data.AUTOTUNE)` always.

**Q: PyTorch vs TensorFlow — how do you choose?**
A: Research/prototyping: PyTorch (dynamic graphs, easier debugging, dominant in academia). Production/deployment: TensorFlow (TFX pipeline, TF Serving, TFLite for mobile). Protein ML: PyTorch dominates (ESM2, AlphaFold2, OpenFold, ProteinMPNN all use PyTorch). TF used at Google DeepMind internally.

### TensorFlow/Keras Time Complexity
| API | Overhead | Use case |
|-----|----------|----------|
| `model.fit()` | Low (optimized) | Standard training |
| `tf.GradientTape` | Medium | Custom training |
| `@tf.function` | JIT compile first call, then fast | Repeated calls |
| `tf.data` pipeline | Near-zero (async) | Large datasets |

In [ ]:
# TensorFlow Tensors & Eager Execution
import numpy as np
try:
    import tensorflow as tf
    print(f"TensorFlow version: {tf.__version__}")

    # TF tensors
    x = tf.constant([[1.0, 2.0], [3.0, 4.0]])
    print(f"Tensor: {x}")
    print(f"Shape: {x.shape}, dtype: {x.dtype}")

    # Eager execution: operations run immediately (like numpy)
    y = tf.matmul(x, x)
    print(f"x @ x = {y.numpy()}")

    # GradientTape: manual gradient computation
    w = tf.Variable([1.0, 2.0, 3.0])
    with tf.GradientTape() as tape:
        y = tf.reduce_sum(w**2)  # y = sum(w^2)
    grad = tape.gradient(y, w)
    print(f"dy/dw = 2*w = {grad.numpy()}")

except ImportError:
    print("TensorFlow not installed. Key concepts:")
    print("  tf.constant: immutable tensor")
    print("  tf.Variable: mutable tensor (trainable weights)")
    print("  tf.GradientTape: records ops for auto-differentiation")
    print("  Eager execution: ops run immediately (no graph construction)")

    # Demonstrate concepts with numpy
    import numpy as np
    w = np.array([1.0, 2.0, 3.0])
    y = np.sum(w**2)
    grad = 2 * w  # dy/dw analytically
    print(f"\nNumpy analog: w={w}, y={y:.1f}, grad={grad}")

## PyTorch ↔ TensorFlow Quick Translation Cheat Sheet

This cheat sheet covers the 80% of operations you'll encounter when reading biology papers. Use it as a reference when porting models between frameworks.

### Data Types
| Concept | PyTorch | TensorFlow |
|---------|---------|------------|
| 32-bit float | `torch.float32` or `torch.float` | `tf.float32` |
| 16-bit float | `torch.float16` | `tf.float16` |
| 64-bit int | `torch.int64` or `torch.long` | `tf.int64` |
| 32-bit int | `torch.int32` | `tf.int32` |
| Boolean | `torch.bool` | `tf.bool` |

### Channels Convention — CRITICAL
```
PyTorch Conv1D: (batch, channels, length)  ← channels FIRST
TF     Conv1D: (batch, length, channels)  ← channels LAST (default)

PyTorch Conv2D: (batch, channels, H, W)   ← NCHW
TF     Conv2D: (batch, H, W, channels)   ← NHWC (default)
```

### Training Mode
```python
# PyTorch                         # TensorFlow
model.train()                     # model(x, training=True)
model.eval()                      # model(x, training=False)
```

### Gradient Handling
```python
# PyTorch                         # TensorFlow
optimizer.zero_grad()             # (tape is cleared automatically after gradient())
loss.backward()                   # grads = tape.gradient(loss, vars)
optimizer.step()                  # optimizer.apply_gradients(zip(grads, vars))
```

In [ ]:
# TensorFlow/Keras summary: key patterns
import numpy as np
try:
    import tensorflow as tf
    print(f"TF version: {tf.__version__}")
    # Simple regression model
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(16, activation='relu', input_shape=(5,)),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    X = np.random.randn(100, 5).astype(np.float32)
    y = X.sum(1, keepdims=True)
    model.fit(X, y, epochs=10, verbose=0)
    print(f"Final loss: {model.evaluate(X, y, verbose=0):.4f}")
except ImportError:
    print("Key Keras patterns:")
    print("  model.fit(X, y, epochs=N, validation_split=0.2, callbacks=[...])")
    print("  model.evaluate(X_test, y_test) → loss, metrics")
    print("  model.predict(X_new) → predictions")
    print("  model.save('model.h5') / keras.models.load_model('model.h5')")

## Mastery Check

Before moving on, make sure you can:
1. explain the core concept of this notebook in plain English without looking
2. write a small toy example from scratch
3. identify one common mistake and explain why it is wrong
4. say whether you should revisit math, Python, or ML basics before continuing


---
## 📚 Resources — TensorFlow, Keras & Framework Fluency

### University Courses (All Free)
| Course | Why This One |
|--------|-------------|
| **[DeepLearning.AI TensorFlow Developer](https://www.coursera.org/professional-certificates/tensorflow-in-practice)** | Andrew Ng's team; free to audit; 4 courses |
| **[MIT 6.S191 Lab 2](https://introtodeeplearning.com/)** | Uses TensorFlow; covers CNNs, RNNs in TF |
| **[Google Machine Learning Crash Course](https://developers.google.com/machine-learning/crash-course)** | 15 hours; uses TF/Keras throughout; free |

### Framework Comparison (Important for Interviews)
| Feature | PyTorch | TensorFlow/Keras |
|---------|---------|-----------------|
| Default in research | ✓ Yes (2023+) | Less common now |
| Default in production | Both | Both (TF Serving) |
| Debugging | Easier (eager) | Harder (graphs) |
| Code style | Pythonic | More declarative |
| Bioinformatics | PyTorch dominant | Some legacy code |

### Key TensorFlow Concepts to Know
- `tf.GradientTape()` — TF's equivalent of PyTorch autograd
- `model.fit()` vs custom training loop — know both
- `tf.data.Dataset` — efficient data pipeline
- `@tf.function` — graph compilation for speed

### Essential Reading
- **[Keras documentation](https://keras.io/guides/)** — the Sequential and Functional API guides
- **[Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow](https://www.oreilly.com/library/view/hands-on-machine-learning/9781492032632/)** (Géron) — Chapters 10-12; most widely used practical ML textbook

### Interview Tip
Most bioinformatics and drug-discovery ML roles in 2024+ use PyTorch. Know TensorFlow conceptually but master PyTorch. When asked "do you know TensorFlow?", the honest answer is: "I work primarily in PyTorch; I can read and adapt TensorFlow code."
